# Simple Agentic Resume Screening with OpenAI

This notebook is a **simplified Python-based Agentic AI version** of your original resume screening code.

It keeps the same core data flow:

1. Extract text from resume  
2. Analyze job description  
3. Compare resume vs job description  
4. Generate score  
5. Suggest improvements  
6. Store final analysis in Chroma vector DB

It is intentionally simpler than the earlier multi-file project, while keeping the **data integrity and core behavior** aligned with your original uploaded code.


## Install dependencies

Run this cell once if the libraries are not installed.


In [ ]:
# Uncomment and run if needed
!pip install -q openai langchain langchain-openai langchain-community langchain-text-splitters chromadb pypdf docx2txt python-dotenv tiktoken


## 1. Imports and environment setup

In [ ]:
import os
import re
from pathlib import Path
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
   raise ValueError("OPENAI_API_KEY not found. Please add it in your .env file or environment variables.")

print("OpenAI API key loaded successfully.")

## 2. LLM and vector store setup

In [ ]:
VECTOR_STORE_DIR = "chroma_store"

def get_llm(temperature=0.2):
    return ChatOpenAI(
        model="gpt-3.5-turbo",
        temperature=temperature,
        api_key=OPENAI_API_KEY
    )

def get_embeddings():
    return OpenAIEmbeddings(
        model="text-embedding-3-small",
        api_key=OPENAI_API_KEY
    )

def get_vectorstore():
    os.makedirs(VECTOR_STORE_DIR, exist_ok=True)
    return Chroma(
        persist_directory=VECTOR_STORE_DIR,
        embedding_function=get_embeddings()
    )

print("LLM and vector store helpers are ready.")

## 3. File loading

In [ ]:
import os
from pathlib import Path

from langchain_community.document_loaders import (
    PyPDFLoader,
    Docx2txtLoader,
    TextLoader
)


def extract_text_from_resume(file_path):
    """
    Extract text from PDF, DOCX, or TXT resume.
    file_path can be a string or Path object.
    """

    try:
        file_path = Path(file_path)

        # ✅ Check if file exists
        if not file_path.exists():
            raise FileNotFoundError(f"File not found: {file_path}")

        ext = file_path.suffix.lower()

        # ✅ Select correct loader
        if ext == ".pdf":
            loader = PyPDFLoader(str(file_path))
        elif ext == ".docx":
            loader = Docx2txtLoader(str(file_path))
        elif ext == ".txt":
            loader = TextLoader(str(file_path), encoding="utf-8")
        else:
            raise ValueError(f"Unsupported file format: {ext}")

        # ✅ Load documents
        docs = loader.load()

        # ✅ Extract text safely
        text = "\n".join(doc.page_content for doc in docs if doc.page_content)

        if not text.strip():
            raise ValueError("No text extracted from file")

        return text

    except Exception as e:
        print(f"❌ Error processing file: {e}")
        return None


# ✅ Example usage (OUTSIDE function)
if __name__ == "__main__":
    resume_text = extract_text_from_resume("Resume_Sample 1.pdf")

    if resume_text:
        print("\n📄 Extracted Resume Text:\n")
        print(resume_text[:1000])  # print first 1000 chars

## 4. Text splitter and vector DB storage

In [ ]:
def split_text(text, chunk_size=500, chunk_overlap=50):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    return splitter.create_documents([text])

def store_analysis(doc_id, analysis_text):
    vectorstore = get_vectorstore()
    docs = split_text(analysis_text)
    ids = [f"{doc_id}_chunk_{i}" for i in range(len(docs))]
    vectorstore.add_documents(docs, ids=ids)
    vectorstore.persist()

def retrieve_similar_analyses(query, k=2):
    vectorstore = get_vectorstore()
    results = vectorstore.similarity_search(query, k=k)
    return "\n\n".join(doc.page_content for doc in results) if results else ""


## 5. Simple agent functions

In [ ]:
def run_resume_reader_agent(resume_text):
    llm = get_llm(temperature=0.1)

    prompt = ChatPromptTemplate(
        input_variables=["resume_text"],
        template="""
You are a Resume Reader Agent.

Read the resume carefully and extract:

1. Candidate Summary
2. Key Skills
3. Experience
4. Education
5. Certifications
6. Projects
7. Tools and Technologies

Resume:
{resume_text}
"""
    )

    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"resume_text": resume_text})


def run_job_analyst_agent(job_requirements):
    llm = get_llm(temperature=0.1)

    prompt = ChatPromptTemplate(
        input_variables=["job_requirements"],
        template="""
You are a Job Requirement Analyst Agent.

Analyze the job requirements and identify:

1. Must-have skills
2. Good-to-have skills
3. Required experience
4. Important tools/technologies
5. Domain expectations

Job Requirements:
{job_requirements}
"""
    )

    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"job_requirements": job_requirements})


def run_matching_agent(resume_summary, job_summary, past_context=""):
    llm = get_llm(temperature=0.2)

    prompt = ChatPromptTemplate(
        input_variables=["resume_summary", "job_summary", "past_context"],
        template="""
You are a Resume Matching Agent.

Compare the candidate profile and the job requirements.

Candidate Profile:
{resume_summary}

Job Requirement Analysis:
{job_summary}

Past Similar Analysis Context:
{past_context}

Provide:
1. Strong matches
2. Partial matches
3. Missing skills
4. Risk areas
5. Final matching summary
"""
    )

    chain = prompt | llm | StrOutputParser()
    return chain.invoke({
        "resume_summary": resume_summary,
        "job_summary": job_summary,
        "past_context": past_context
    })


def run_scoring_agent(match_output):
    llm = get_llm(temperature=0.1)

    prompt = ChatPromptTemplate(
        input_variables=["match_output"],
        template="""
You are a Resume Scoring Agent.

Based on the match analysis below, score the candidate.

Match Analysis:
{match_output}

Return exactly in this style:

Technical Fit: XX/100
Experience Fit: XX/100
Tool Fit: XX/100
Domain Fit: XX/100
Overall Suitability Score: XX%
Final Verdict: Strong Fit / Moderate Fit / Weak Fit
"""
    )

    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"match_output": match_output})


def run_improvement_agent(match_output):
    llm = get_llm(temperature=0.2)

    prompt = ChatPromptTemplate(
        input_variables=["match_output"],
        template="""
You are a Resume Improvement Agent.

Using the analysis below, suggest:

1. Missing skills to learn
2. Resume improvements
3. ATS keywords to add
4. Interview preparation suggestions

Analysis:
{match_output}
"""
    )

    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"match_output": match_output})


## 6. Score extractor

In [ ]:
def extract_suitability_score(text):
    match = re.search(r"Overall Suitability Score:\s*(\d{1,3})%", text)
    if match:
        return int(match.group(1))
    return None


## 7. Main workflow coordinator

In [ ]:
def run_agentic_resume_screening(resume_file_path, job_requirements):
    """
    Simple agentic workflow:
    1. Extract resume text
    2. Resume reader agent
    3. Job analyst agent
    4. Retrieve similar analyses from Chroma
    5. Matching agent
    6. Scoring agent
    7. Improvement agent
    8. Store final report
    """
    resume_text = extract_text_from_resume(resume_file_path)
    doc_id = Path(resume_file_path).stem

    resume_summary = run_resume_reader_agent(resume_text)
    job_summary = run_job_analyst_agent(job_requirements)

    past_context = retrieve_similar_analyses(job_requirements, k=2)

    match_output = run_matching_agent(
        resume_summary=resume_summary,
        job_summary=job_summary,
        past_context=past_context
    )

    score_output = run_scoring_agent(match_output)
    improvement_output = run_improvement_agent(match_output)

    final_report = f"""
# Final Resume Screening Report

## 1. Resume Summary
{resume_summary}

---

## 2. Job Requirement Summary
{job_summary}

---

## 3. Matching Analysis
{match_output}

---

## 4. Scoring
{score_output}

---

## 5. Improvement Suggestions
{improvement_output}
"""

    store_analysis(doc_id, final_report)

    return {
        "resume_text": resume_text,
        "resume_summary": resume_summary,
        "job_summary": job_summary,
        "match_output": match_output,
        "score_output": score_output,
        "improvement_output": improvement_output,
        "final_report": final_report,
        "score": extract_suitability_score(score_output)
    }


## 8. Run the project

In [ ]:
# Example inputs
resume_file_path = "C:\\Gen AI Projects\\Agentic AI\\LangChain Project\\Resume_Sample 1.pdf"  # change this to your file path

job_requirements = """
We are hiring a Python AI Engineer with experience in:
- Python
- APIs
- OpenAI or LLM integration
- Vector databases
- Streamlit or FastAPI
- Prompt engineering
- RAG or agentic workflows
- Good communication and problem-solving skills
"""

# Run only after updating the resume_file_path
result = run_agentic_resume_screening(resume_file_path, job_requirements)


## 9. View outputs

In [ ]:
# Uncomment after running the workflow

print(result["final_report"])
print("\nExtracted Score:", result["score"])


## 10. Save report to a text file

In [ ]:
# Uncomment after running the workflow

# with open("agentic_resume_report.txt", "w", encoding="utf-8") as f:
#     f.write(result["final_report"])
#
# print("Report saved as agentic_resume_report.txt")
